# SDL catalogue manager — persons, organizations & projects

A standalone tool to **browse, search and register** the three SDL catalogues that an experiment
references: **persons** (authors), **organizations** (institutions) and **projects**. Use it to find
the numeric **ids** you need for a manifest, and to **create** an entry when it is missing — the same
catalogues the publisher resolves `authors / organizations / projects` against.

**Flow:** *Setup → Login → use the tool*. In the tool, pick a catalogue, type a substring and **Search**
(leave the filter empty to **list everything**); if nothing matches, fill the *Create a new entry*
fields and **Create entry** to register it and get its id.

> Read & create only. There is **no delete** here on purpose — removing catalogue entries (or
> experiments) is destructive and out of scope, exactly as in `sdl_lib`.
>
> Creating writes to the catalogue of the server you logged into: harmless on **dev**, a real new
> entry on **prod**.


## 1. Setup & dependencies

In [ ]:
# --- Setup: load the companion library sdl_lib.py and check dependencies -----
import os, sys, importlib, importlib.util
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# Locate sdl_lib.py: it lives either in a sibling 01_sdl_publish/ folder (repo
# layout) or directly in the parent folder (server ~/SDL layout). Search both.
_cands = [os.path.join(os.getcwd(), "..", "01_sdl_publish"),
          os.path.join(os.getcwd(), ".."),
          os.getcwd(),
          os.path.expanduser("~/HySEALab/Notebooks/01_sdl_publish"),  # HySEALab menu: shared lib
          os.path.expanduser("~/HySEALab/Notebooks"),                 # shared (next to map viewer)
          os.path.expanduser("~/SDL")]                                # personal working copy
LIB_DIR = next((os.path.abspath(d) for d in _cands
                if os.path.exists(os.path.join(d, "sdl_lib.py"))), None)
if LIB_DIR is None:
    raise FileNotFoundError("sdl_lib.py not found in ../01_sdl_publish, .. or .")
sys.path.insert(0, LIB_DIR)

REQUIRED = {
    "requests": "requests", "requests_toolbelt": "requests-toolbelt",
    "rasterio": "rasterio", "cmocean": "cmocean",
    "matplotlib": "matplotlib", "numpy": "numpy", "tqdm": "tqdm",
    "yaml": "pyyaml", "jsonschema": "jsonschema",
}
_missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]

_dep_out = widgets.Output()
def _install_missing(_):
    import subprocess
    with _dep_out:
        clear_output()
        print("Installing into the user site:", " ".join(_missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "--user", *_missing])
        print("Done. Restart the kernel and re-run this cell.")
_btn_install = widgets.Button(description="Install missing deps", button_style="warning")
_btn_install.on_click(_install_missing)

if _missing:
    print("Missing dependencies:", ", ".join(_missing))
    display(_btn_install, _dep_out)
else:
    print("All dependencies available.")

import sdl_lib
importlib.reload(sdl_lib)
print("sdl_lib loaded from:", LIB_DIR)

# Shared state for this notebook (login token lives only in memory).
S = {"base_url": "https://sdl-dev.hpc.cineca.it", "token": None, "headers": None}
print("Setup ready.")


## 2. Login

In [ ]:
# --- Login: authenticate once; the token is reused by the publisher ----------
w_env = widgets.Dropdown(
    options=[("Development (sdl-dev.hpc.cineca.it)", "https://sdl-dev.hpc.cineca.it"),
             ("Production  (sdl.hpc.cineca.it)", "https://sdl.hpc.cineca.it")],
    value="https://sdl-dev.hpc.cineca.it", description="Server:",
    layout=widgets.Layout(width="440px"))
w_user = widgets.Text(description="Username:", placeholder="your SDL e-mail",
                      layout=widgets.Layout(width="440px"))
w_pass = widgets.Password(description="Password:", layout=widgets.Layout(width="440px"))
w_login = widgets.Button(description="Login", button_style="primary")
out_login = widgets.Output()

def _do_login(_):
    with out_login:
        clear_output()
        S["base_url"] = w_env.value
        u = w_user.value.strip()
        if not u or not w_pass.value:
            print("Enter username and password."); return
        try:
            tok = sdl_lib.get_auth_token(S["base_url"], u, w_pass.value)
        except Exception as e:
            print("Login failed:", e); return
        if not tok:
            print("Login failed: no token returned."); return
        S["token"] = tok
        S["headers"] = sdl_lib.auth_headers(tok)
        print("Login OK - token stored. Server:", S["base_url"])
w_login.on_click(_do_login)
display(widgets.VBox([w_env, w_user, w_pass, w_login, out_login]))


## 3. Browse / search / create

Empty filter + **Search** lists the whole catalogue. The ids shown go straight into a manifest as
`authors: [{id: <n>}]`, `organizations: [{id: <n>}]` or `projects: [{id: <n>}]` — or feed the
interactive builder (`build_manifest_interactive_EN.ipynb`).

In [ ]:
# --- (Optional) Catalogue tool: SEARCH any catalogue, or CREATE a new entry --
# A new experiment references authors (>=1 required) and, optionally, organizations
# and projects. Search the three SDL catalogues to get numeric ids for the manifest
# ({id: N}); if someone/something is missing, create it here and reuse the new id.
# (Alternative: put {name: ...} in the manifest and the publisher creates it at
# publish time.)  NOTE: creating writes to the catalogue of the server you are
# logged into - harmless on dev, a real new entry on prod.
cat_kind = widgets.Dropdown(
    options=[("Persons (authors)", "persons"),
             ("Organizations", "organizations"),
             ("Projects", "projects")],
    value="persons", description="Catalogue:", layout=widgets.Layout(width="330px"))
cat_q = widgets.Text(description="Filter:", placeholder="substring; empty = list all",
                     layout=widgets.Layout(width="450px"))
cat_btn = widgets.Button(description="Search", button_style="info")
# creation fields - their labels/visibility adapt to the chosen catalogue (see _on_kind below)
_fstyle = {"description_width": "95px"}
n1 = widgets.Text(layout=widgets.Layout(width="470px"), style=_fstyle)
n2 = widgets.Text(layout=widgets.Layout(width="470px"), style=_fstyle)
n3 = widgets.Text(layout=widgets.Layout(width="470px"), style=_fstyle)
new_btn = widgets.Button(description="Create entry", button_style="success")
cat_out = widgets.Output()

def _on_kind(_=None):
    # Persons need given/family/email; organizations only a legalName; projects only a title.
    kind = cat_kind.value
    if kind == "persons":
        n1.description, n1.placeholder = "Given name:", "e.g. Jorge"
        n2.description, n2.placeholder = "Family name:", "e.g. Macias (required)"
        n3.description, n3.placeholder = "Email:", "optional"
        n2.layout.display = n3.layout.display = ""
    else:
        n1.description = "legalName:" if kind == "organizations" else "title:"
        n1.placeholder = ("institution legal name, e.g. University of Malaga"
                          if kind == "organizations" else "project title, e.g. Geo-INQUIRE")
        n2.layout.display = n3.layout.display = "none"   # not used for orgs/projects
        n2.value = n3.value = ""
cat_kind.observe(_on_kind, names="value")
_on_kind()   # set the initial (persons) layout

def _label(kind, o):
    if kind == "persons":
        e = o.get("emails"); em = e[0] if isinstance(e, list) and e else (o.get("email") or "")
        return (f"{o.get('givenName','')} {o.get('familyName','')}".strip()
                + (f"  <{em}>" if em else ""))
    if kind == "organizations":
        return o.get("legalName") or o.get("name") or ""
    return o.get("title") or o.get("name") or ""

def _fetch(kind):
    return (sdl_lib.list_persons(S["base_url"], S["headers"]) if kind == "persons"
            else sdl_lib.list_catalog(S["base_url"], S["headers"], kind))

def _search(_):
    with cat_out:
        clear_output()
        if not S.get("token"):
            print("Login first (section 2)."); return
        kind = cat_kind.value
        items = _fetch(kind)
        q = cat_q.value.strip().lower()
        rows = [(o.get("id"), _label(kind, o)) for o in items]
        if q:
            rows = [(i, l) for i, l in rows if q in l.lower()]
        if not rows:
            print(f"No {kind} match '{q}'. Total: {len(items)}. Use 'Create entry' to add it.")
            return
        print(f"{kind}: {len(rows)} match(es). In the manifest: {kind}: [{{id: <n>}}]\n")
        for i, l in sorted(rows, key=lambda r: str(r[1]).lower()):
            print(f"  id={str(i):>5}  {l}")
cat_btn.on_click(_search)

def _create(_):
    with cat_out:
        clear_output()
        if not S.get("token"):
            print("Login first (section 2)."); return
        kind = cat_kind.value
        try:
            if kind == "persons":
                if not n2.value.strip():
                    print("Family name is required for a person."); return
                sdl_lib.create_person(S["base_url"], S["headers"],
                                      n1.value.strip(), n2.value.strip(), n3.value.strip())
                match = lambda o: (o.get("givenName", "") == n1.value.strip()
                                   and o.get("familyName", "") == n2.value.strip())
            else:
                if not n1.value.strip():
                    print("A name is required."); return
                sdl_lib.create_catalog_item(S["base_url"], S["headers"], kind, n1.value.strip())
                nk = {"organizations": "legalName", "projects": "title"}[kind]
                match = lambda o: (o.get(nk) or o.get("name")) == n1.value.strip()
            new = [o for o in _fetch(kind) if match(o)]   # re-list to confirm + read the new id
            if new:
                print("Created. New " + kind[:-1] + " id(s): "
                      + ", ".join(str(o.get("id")) for o in new))
                print(f"-> put  {kind}: [{{id: {new[-1].get('id')}}}]  in the manifest.")
            else:
                print("Created, but could not read the new id back; click Search to find it.")
        except Exception as e:
            print("Create failed:", type(e).__name__, e)
new_btn.on_click(_create)

display(widgets.VBox([cat_kind,
                      widgets.HBox([cat_q, cat_btn]),
                      widgets.HTML("<b>Create a new entry</b> (only if Search finds nothing):"),
                      n1, n2, n3, new_btn, cat_out]))


## Notes

- The three catalogues are reached with `sdl_lib.list_persons` and `sdl_lib.list_catalog(...,
  "organizations"|"projects")`; new entries with `sdl_lib.create_person` and
  `sdl_lib.create_catalog_item`.
- **Ids differ between dev and prod** (separate catalogues), so look them up on the same server you will
  publish to.
- Alternatively you can skip ids entirely and write `{name: ...}` in the manifest; the publisher then
  searches by name and **creates the entry at publish time** if missing.
